# 05 · CNN + STN + Channel Attention (Hannah)

Proposed model: **STN → CNN with channel attention**.

The STN (Hannah) is placed in front of Alex's attention-augmented CNN (`CNNAttention`), so the combined model is exactly the *CNN + attention* ablation variant plus the same STN used in `04`.

Compare against: basic CNN, CNN + STN (`04`), CNN + attention (Alex), and ResNet-18 (`03`).

In [ ]:
# --- Locate the project root so `src/` imports work from any folder ---
import sys, os
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
import torch
from src.config import TrainConfig, get_device
from src.pipeline import (set_seed, fit, evaluate, plot_history, save_run_summary,
                          make_fake_loaders)
from src.models.stn import STN, STNClassifier, visualize_stn
from src.config import FIGURES_DIR

cfg = TrainConfig()          # shared config -> fair ablation comparison
set_seed(cfg.seed)
device = get_device()
print("Device:", device)

In [ ]:
# ----------------------------------------------------------------------------
# ALEX'S PART: channel-attention blocks and the CNN that uses them.
# Expected: src/models/cnn_attention.py with CNNAttention(num_classes) -> nn.Module
# (Hannah does not implement the attention blocks.)
# ----------------------------------------------------------------------------
try:
    from src.models.cnn_attention import CNNAttention    # ALEX'S PART
except ImportError:
    CNNAttention = None
    print("Alex's CNNAttention is not in the repo yet. Merge his src/models/cnn_attention.py before building the model.")

In [ ]:
# ----------------------------------------------------------------------------
# DATA  --  ALEX'S PART (GTSRB download, preprocessing, train/val/test splits)
# ----------------------------------------------------------------------------
USE_FAKE_DATA = True   # <-- set to False once Alex's data pipeline is merged

if USE_FAKE_DATA:
    # Random data: only verifies the code runs. Results are meaningless.
    train_loader, val_loader, test_loader = make_fake_loaders(cfg)
else:
    # ALEX'S PART: assumed interface (see notebook 00). Adjust the call if Alex's differs.
    from src.data import get_dataloaders
    train_loader, val_loader, test_loader = get_dataloaders(
        batch_size=cfg.batch_size, num_workers=cfg.num_workers, img_size=cfg.img_size)

In [ ]:
assert CNNAttention is not None, "Needs Alex's CNNAttention (src/models/cnn_attention.py)."

stn = STN(in_channels=cfg.in_channels, img_size=cfg.img_size)          # Hannah
classifier = CNNAttention(num_classes=cfg.num_classes)                  # ALEX'S PART
model = STNClassifier(stn, classifier)

n_total = sum(p.numel() for p in model.parameters())
n_stn = sum(p.numel() for p in stn.parameters())
print(f"Total parameters: {n_total:,}  (STN adds {n_stn:,})")

In [ ]:
history = fit(model, train_loader, val_loader, cfg, run_name="cnn_stn_attention", device=device)
plot_history(history, title="cnn_stn_attention")

In [ ]:
val_metrics = evaluate(model, val_loader, device, cfg.num_classes)
print({k: round(v, 4) for k, v in val_metrics.items() if isinstance(v, float)})
save_run_summary("cnn_stn_attention", cfg, val_metrics, split="val")

EVAL_ON_TEST = False   # flip only for the final results, after all tuning is done
if EVAL_ON_TEST:
    test_metrics = evaluate(model, test_loader, device, cfg.num_classes)
    save_run_summary("cnn_stn_attention", cfg, test_metrics, split="test")
    print({k: round(v, 4) for k, v in test_metrics.items() if isinstance(v, float)})

# ALEX'S PART: corrupted-image evaluation (fog, rain, blur, noise, brightness, occlusion; severity 1-5)
# is implemented by Alex. Once his corrupted test loaders exist, call evaluate(model, corrupted_loader, ...) here.

In [ ]:
# What did the STN learn? (top: input, bottom: after STN)
visualize_stn(model, val_loader, device, n=8, save_path=FIGURES_DIR / "cnn_stn_attention_stn_examples.png")
print("Example theta (first image):\n", model.stn.last_theta[0].cpu())